# 4단계 · 직전 N쇼트 이탈도 (라벨 없이 점수 매기기)

**하는 일**: 각 쇼트가 "바로 앞 N쇼트의 평소 모습"에서 얼마나 벗어났는지를 점수로 만듭니다.
라벨(PassOrFail)은 점수 만드는 데 **전혀 쓰지 않고**, 채점할 때만 씁니다.

**왜 하나**: 3단계에서 지도학습 랜덤포레스트는 이웃 쇼트를 학습에서 떼어 놓으면(간격 두기) 무작위 수준이었습니다.
라벨을 외울 수 없는 방법으로도 불량이 위로 올라오는지 확인합니다.

진행: 각 질문 칸에 **실행 전에 예상을 먼저 적고**, 실행한 뒤 맞았는지 확인하세요.

In [ ]:
import sys; sys.path.append("..")
import numpy as np, pandas as pd
from src.official import *

DATA = {p: load(p)[0] for p in ["CN7", "RG3"]}      # 라벨 파일만 씀 (비라벨은 따로 표준화돼 섞을 수 없음)
CEIL = {p: round(float(f1_ceiling(DATA[p])["F1"]), 3) for p in DATA}
print("이론 상한 F1:", CEIL)

### 점수 만드는 법
1. 쌍(같은 쇼트의 LH/RH)은 값이 같으니 **쇼트 하나당 한 행**으로 줄입니다. 그래야 같은 쇼트가 '평소 모습'에 두 번 들어가지 않습니다.
2. 쇼트 i 앞의 N쇼트에서 변수별 **중앙값**과 **퍼짐**(MAD×1.4826과 표준편차 중 큰 값)을 구합니다.
3. 변수마다 `z = |지금 값 − 중앙값| / 퍼짐` 을 계산합니다. 앞 N쇼트 동안 값이 전혀 안 변한 변수(설정값)는 0으로 둡니다.
4. 24개 z를 하나로 합칩니다. `rms`(제곱평균의 제곱근: 전체적으로 벗어남) / `max`(가장 많이 벗어난 변수 하나)
5. 쇼트 점수를 그 쇼트의 두 행(LH, RH)에 똑같이 붙입니다.

> **질문 4-1.** 이 점수는 라벨을 안 쓰므로 교차검증으로 fold를 나눌 필요가 없습니다. 왜 그런가요?
> 그래도 **결과를 보고 N이나 rms/max를 고르면** 남는 문제가 있습니다. 무엇일까요?
>
> 내 예상:

In [ ]:
def shot_table(lab):
    """쌍을 쇼트 하나로 줄인 표 (index = pair)."""
    return lab.drop_duplicates("pair").set_index("pair")[VARS]


def robust_z(x, P):
    """x(한 쇼트)가 P(기준 쇼트들)에서 변수별로 몇 '퍼짐'만큼 떨어졌는지. 최대 20으로 자름."""
    med = np.median(P, axis=0)
    sd = np.maximum(np.median(np.abs(P - med), axis=0) * 1.4826, P.std(axis=0))
    ok = sd > 1e-6
    return np.where(ok, np.minimum(np.abs(x - med) / np.where(ok, sd, 1), 20), 0)


AGG = {"rms": lambda z: np.sqrt((z ** 2).mean()), "max": lambda z: z.max()}


def prev_dev(lab, N=10, agg="rms", perm=None):
    """직전 N쇼트 이탈도. perm 을 주면 쇼트 순서를 그 순서로 바꾼 뒤 계산(순서 섞기 실험용).
    돌려주는 것: (행별 점수, 행별 '가장 많이 벗어난 변수')"""
    sh = shot_table(lab)
    if perm is not None:
        sh = sh.iloc[perm]
    X = sh.values
    sc = np.zeros(len(X))
    top = np.full(len(X), "", dtype=object)
    for i in range(len(X)):
        P = X[max(0, i - N): i]
        if len(P) < 3:          # 앞 쇼트가 3개 미만이면 기준을 못 세움 → 0점
            continue
        z = robust_z(X[i], P)
        sc[i] = AGG[agg](z)
        top[i] = VARS[int(z.argmax())]
    return (lab["pair"].map(pd.Series(sc, index=sh.index)).values,
            lab["pair"].map(pd.Series(top, index=sh.index)).values)

> **질문 4-2.** CN7과 RG3 중 이 점수가 더 잘 통할 쪽은? 이유는?
> (힌트: 1단계 표. 쌍둥이 불량이 CN7은 11/17, RG3는 25/25. 쌍둥이 두 행은 이 점수도 똑같이 받습니다.)
>
> 내 예상:

In [ ]:
rows = []
for p, lab in DATA.items():
    for N in [5, 10, 30]:
        for agg in ["rms", "max"]:
            s, _ = prev_dev(lab, N, agg)
            rows.append(evaluate(lab, s, f"직전{N}쇼트·{agg}", p, CEIL[p]))
T4 = pd.DataFrame(rows)
cols = ["모델", "제품", "AP", "무작위 평균 대비", "무작위 상단(97.5%)", "유의", "상위1%", "상위5%", "상위10%"]
T4[cols].round(3)

여섯 조합을 **전부** 표에 남기세요. 가장 좋은 하나만 골라 보고하면 4-1에서 말한 문제가 생깁니다.
아래부터는 미리 정해 둔 기본값 **N=10, rms**로 봅니다.

### 순서 섞기 실험
쇼트 순서를 무작위로 섞으면 '직전 N쇼트'는 시간과 상관없는 '아무 N쇼트'가 됩니다.
**시간 흐름**을 써서 점수가 나온 것이라면, 섞었을 때 점수가 떨어져야 합니다.

> **질문 4-3.** CN7에서 순서를 섞으면 AP가 (떨어진다 / 비슷하다 / 오른다) 중 무엇일까요?
>
> 내 예상:

In [ ]:
for p, lab in DATA.items():
    s, _ = prev_dev(lab)
    ap0 = evaluate(lab, s, "-", p, CEIL[p])["AP"]
    n_shot = lab["pair"].nunique()
    aps = [evaluate(lab, prev_dev(lab, perm=np.random.RandomState(k).permutation(n_shot))[0], "-", p, CEIL[p])["AP"]
           for k in range(20)]
    print(f"{p}: 시간 순서 AP {ap0:.3f}  |  순서 섞기 20회 AP 평균 {np.mean(aps):.3f} ({min(aps):.3f}~{max(aps):.3f})")

### 파일 전체를 기준으로 하면?
'직전 N쇼트' 대신 **그 제품 라벨 파일 전체 쇼트**의 중앙값·퍼짐을 기준으로 삼아 봅니다.
(라벨은 여전히 안 씁니다.)

> **질문 4-4.** 4-3 결과를 보고, 전체 기준 점수가 직전 10쇼트 점수보다 높을지 낮을지 예상해 보세요.
>
> 내 예상:

In [ ]:
def global_dev(lab, agg="rms"):
    """파일 전체 쇼트를 기준으로 한 이탈도."""
    sh = shot_table(lab)
    X = sh.values
    sc = np.array([AGG[agg](robust_z(x, X)) for x in X])
    return lab["pair"].map(pd.Series(sc, index=sh.index)).values

rows = []
for p, lab in DATA.items():
    rows.append(evaluate(lab, prev_dev(lab)[0], "직전10쇼트·rms", p, CEIL[p]))
    rows.append(evaluate(lab, global_dev(lab), "전체기준·rms", p, CEIL[p]))
pd.DataFrame(rows)[cols].round(3)

### 불량 하나하나 보기
불량 행마다 **쌍 종류**와 **쇼트 순위**(위에서 몇 %인지, 작을수록 의심)를 봅니다.
- 단독: 짝 없는 행 / 쌍둥이: 한쪽만 불량 / 둘다불량: 좌우 모두 불량

> **질문 4-5.** 어떤 쌍 종류의 불량이 위로 올라올까요? 1단계 상한 계산(TP 17 = 둘다불량 6 + 쌍둥이 11)과 연결해서 생각해 보세요.
>
> 내 예상:

In [ ]:
def defect_table(lab, s, top=None):
    shot_s = pd.Series(s, index=lab.index).groupby(lab["pair"]).first()
    n_bad = lab.groupby("pair")["y"].transform("sum")
    n = lab.groupby("pair")["y"].transform("size")
    out = lab.assign(점수=s,
                     쌍=np.where(n == 1, "단독", np.where(n_bad == 2, "둘다불량", "쌍둥이")),
                     **{"쇼트 순위%": [(shot_s > v).mean() * 100 for v in s]})
    if top is not None:
        out["가장 벗어난 변수"] = top
    return out[out["y"] == 1].drop(columns=VARS + ["key", "y"])

for p, lab in DATA.items():
    s, top = prev_dev(lab)
    g = global_dev(lab)
    t = defect_table(lab, g).rename(columns={"점수": "전체기준", "쇼트 순위%": "전체기준 순위%"})
    t["직전10 순위%"] = defect_table(lab, s)["쇼트 순위%"].values
    t["직전10 최대이탈 변수"] = defect_table(lab, s, top)["가장 벗어난 변수"].values
    print(f"── {p} ──")
    display(t.round(2))
    print(t.groupby("쌍")[["전체기준 순위%", "직전10 순위%"]].median().round(1), "\n")

### 날짜 함정 점검 [외부 정보 사용]
외부 1차 파일(labeled_data.csv)의 계획일로 확인한 사실: 공식 CN7 라벨 파일의 **order 0~374 = 10/16**, **375~1210 = 10/20**.
CN7 불량 17건은 **전부 10/16**에 있습니다.

'전체 기준' 점수는 두 날짜를 섞어 기준을 세웁니다. 10/16이 원래 10/20보다 들쭉날쭉한 날이었다면,
불량이라서가 아니라 **10/16이라서** 점수가 높게 나왔을 수 있습니다.

> **질문 4-6.** 10/16 안에서만 기준을 세우면 AP는 어떻게 될까요?
> 그리고 '둘다불량' 3쇼트(6행)를 빼면?
>
> 내 예상:

In [ ]:
lab = DATA["CN7"]
g = global_dev(lab)
d16 = (lab["order"] < 375).values
print(f"전체기준 점수 평균: 10/16 {g[d16].mean():.2f}  /  10/20 {g[~d16].mean():.2f}")
print("불량 수: 10/16", int(lab["y"][d16].sum()), " / 10/20", int(lab["y"][~d16].sum()))

day = lab[d16].reset_index(drop=True)
r = evaluate(day, global_dev(day), "10/16 안에서 전체기준", "CN7", CEIL["CN7"])
print(f"10/16 안에서만: AP {r['AP']:.3f} (무작위 상단 {r['무작위 상단(97.5%)']:.3f}, 유의 {r['유의']}), 상위5% {r['상위5%']:.2f}")

both = (day.groupby("pair")["y"].transform("sum") == 2).values
rest = day[~both].reset_index(drop=True)
r = evaluate(rest, global_dev(rest), "둘다불량 제외", "CN7", CEIL["CN7"])
print(f"둘다불량 3쇼트 빼고: AP {r['AP']:.3f} (무작위 상단 {r['무작위 상단(97.5%)']:.3f}, 유의 {r['유의']})")

### 4단계 정리 (직접 적기)
1. 라벨 없는 이탈도가 잡는 불량은 어떤 불량인가?
2. 시간 흐름(직전 N쇼트)이 도움이 되었나? 그 이유는?
3. 정보 상한(1단계)과 이 결과는 어떻게 이어지나?